# Managing Resources: Mappings, Snapshots, and Instances

## Overview

Learn the fundamental operations for managing Graph OLAP resources. This notebook covers CRUD (Create, Read, Update, Delete) operations for mappings, snapshots, and instances.

**What you'll learn:**
- Initialize the SDK client with authentication
- Create, list, update, and delete mappings
- Export snapshots from your data warehouse
- Launch and manage graph instances
- Connect to instances for querying

**Prerequisites:**
- Completed: [01_core_health_checks.ipynb](./01_core_health_checks.ipynb)
- Stack must be healthy (smoke tests pass)

**Time to complete:** 15 minutes

---

**Test Coverage:**
- Client initialization and context managers
- Mappings API (list, get, create, update, delete, versions)
- Snapshots API (list, get, update, lifecycle)
- Instances API (list, get, update, terminate, lifecycle)
- Instance connection setup

In [ ]:
# Parameters cell - not needed with new test API
# The notebook.test() function handles auth via TestPersona
pass

## Setup

### Test Data Strategy

This test creates its own test data dynamically via SDK:
1. **Base Mapping**: Created at start with Person/KNOWS schema
2. **Base Snapshot**: Created from base mapping
3. **Base Instance**: Created from base snapshot
4. **Inline CRUD**: Additional tests create/delete within each section
5. **Cleanup**: Automatic cleanup via cleanup framework

In [ ]:
import sys

print(f"Python version: {sys.version}")

In [ ]:
from graph_olap import GraphOLAPClient, notebook
from graph_olap.exceptions import NotFoundError
from graph_olap.models.mapping import EdgeDefinition, NodeDefinition, PropertyDefinition
from graph_olap.testing import TestPersona
from graph_olap_schemas import WrapperType

print("SDK imports successful")

# Wake up Starburst Galaxy cluster (auto-suspends after 5 min idle)
notebook.wake_starburst()

## 1. Client Initialization Tests

In [ ]:
# Test: notebook.test() creates test context with automatic cleanup
ctx = notebook.test("CrudTest", persona=TestPersona.ANALYST_ALICE)
client = ctx.client  # Access underlying client for direct SDK operations

assert client is not None, "Client should not be None"
print(f"CRUD 1.1 PASSED: Test context created with persona {TestPersona.ANALYST_ALICE.value.name}")
print(f"  API URL: {client._config.api_url}")

In [ ]:
# Test: Client has resource managers
assert hasattr(client, 'mappings'), "Client should have mappings resource"
assert client.mappings is not None, "mappings should not be None"

assert hasattr(client, 'instances'), "Client should have instances resource"
assert client.instances is not None, "instances should not be None"

assert hasattr(client, 'snapshots'), "Client should have snapshots resource"
assert client.snapshots is not None, "snapshots should not be None"

print("CRUD 1.2 PASSED: All resource managers available")

In [ ]:
# Test: Getting other personas for multi-user testing
# Bob is another analyst we can use for authorization tests
bob_client = ctx.with_persona(TestPersona.ANALYST_BOB)
mappings = bob_client.mappings.list()
assert mappings is not None

print("CRUD 1.3 PASSED: Multi-persona client access verified")

In [ ]:
# Test: Can list resources through SDK
assert client.mappings.list() is not None
assert client.instances.list() is not None
assert client.snapshots.list() is not None

print("CRUD 1.4 PASSED: Resource listing verified")

## Create Base Test Data

Create mapping and instance for subsequent tests.

> **NOTE: SNAPSHOT FUNCTIONALITY DISABLED**
> 
> Explicit snapshot creation has been disabled. Instances are now created directly 
> from mappings using `create_from_mapping()` without requiring explicit snapshots.
> The snapshot-related tests in this notebook have been commented out or skipped.
> See `test_create_from_mapping.py` for the new instance creation workflow.

In [ ]:
# Define Person node with real SQL that works with test data
base_person_node = NodeDefinition(
    label="Person",
    sql="SELECT DISTINCT id, name, age FROM bigquery.graph_olap_e2e.persons",
    primary_key={"name": "id", "type": "STRING"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
        PropertyDefinition(name="age", type="INT64"),
    ]
)

# Define KNOWS edge (uses friendships table)
base_knows_edge = EdgeDefinition(
    type="KNOWS",
    from_node="Person",
    to_node="Person",
    sql="SELECT DISTINCT from_id, to_id, since FROM bigquery.graph_olap_e2e.friendships",
    from_key="from_id",
    to_key="to_id",
    properties=[
        PropertyDefinition(name="since", type="INT64"),
    ]
)

print(f"Test context run ID: {ctx.run_id}")
print(f"Node definitions ready")

## Start Cleanup Context Manager

All resources created from here will be automatically cleaned up.

In [ ]:
# Cleanup is automatic with notebook.test()!
# Resources created via ctx.mapping(), ctx.snapshot(), ctx.instance() are auto-tracked
# and cleaned up on exit (via atexit handler)
print("Starting CRUD E2E Test - resources will be cleaned up automatically via atexit")

## 1.5 Create Base Test Resources

Create a mapping, snapshot, and instance that subsequent tests will use.

In [ ]:
# Create base mapping using ctx.mapping() (auto-named, auto-tracked)
BASE_MAPPING = ctx.mapping(
    name=f"CrudTest-Base-{ctx.run_id}",
    description="Base mapping for CRUD E2E tests",
    node_definitions=[base_person_node],
    edge_definitions=[base_knows_edge],
)
BASE_MAPPING_ID = BASE_MAPPING.id
BASE_MAPPING_NAME = BASE_MAPPING.name

print(f"Created base mapping: {BASE_MAPPING_NAME} (id={BASE_MAPPING_ID})")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# This cell tests explicit snapshot creation which has been disabled.
# Instances are now created directly from mappings without explicit snapshots.
# Keeping the cell commented out for reference.

# # Create base snapshot using ctx.snapshot() (auto-named, auto-tracked)
# print("Creating snapshot and waiting for export...")
# BASE_SNAPSHOT = ctx.snapshot(BASE_MAPPING, timeout=180)
# BASE_SNAPSHOT_ID = BASE_SNAPSHOT.id
# BASE_SNAPSHOT_NAME = BASE_SNAPSHOT.name
# 
# print(f"Created base snapshot: {BASE_SNAPSHOT_NAME} (id={BASE_SNAPSHOT_ID}, status={BASE_SNAPSHOT.status})")

# Instead, we'll create an instance directly from the mapping
print("SNAPSHOT FUNCTIONALITY DISABLED - Skipping explicit snapshot creation")
print("Instances are now created directly from mappings via create_from_mapping()")

In [ ]:
# SNAPSHOT FUNCTIONALITY DISABLED
# This cell previously created an instance from a snapshot.
# Now we create instances directly from mappings using create_from_mapping().

from graph_olap_schemas import WrapperType

print("Creating instance directly from mapping (bypasses explicit snapshot)...")
BASE_INSTANCE = client.instances.create_from_mapping_and_wait(
    mapping_id=BASE_MAPPING_ID,
    name=f"CrudTest-Instance-{ctx.run_id}",
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=300,  # Longer timeout since snapshot export happens automatically
    poll_interval=5,
)
BASE_INSTANCE_ID = BASE_INSTANCE.id
BASE_INSTANCE_NAME = BASE_INSTANCE.name

# Track for cleanup
ctx._track('instance', BASE_INSTANCE_ID, BASE_INSTANCE_NAME)

print(f"Created base instance: {BASE_INSTANCE_NAME} (id={BASE_INSTANCE_ID}, status={BASE_INSTANCE.status})")

# Get the auto-created snapshot info for tests that need it
# Note: The snapshot was created automatically by create_from_mapping
BASE_SNAPSHOT_ID = BASE_INSTANCE.snapshot_id
BASE_SNAPSHOT = client.snapshots.get(BASE_SNAPSHOT_ID) if BASE_SNAPSHOT_ID else None
BASE_SNAPSHOT_NAME = BASE_SNAPSHOT.name if BASE_SNAPSHOT else "N/A"
print(f"Auto-created snapshot: {BASE_SNAPSHOT_NAME} (id={BASE_SNAPSHOT_ID})")

In [ ]:
# Connection helper not needed - SDK handles it
print("Setup complete - ready for connection tests")

## 2. Mappings API Tests

In [ ]:
# Test: List mappings returns results
mappings = client.mappings.list()

assert len(mappings) >= 1, f"Expected at least 1 mapping, got {len(mappings)}"
print(f"CRUD 2.1 PASSED: Found {len(mappings)} mapping(s)")

# Test: Our base mapping is in list
mapping_ids = [m.id for m in mappings]
assert BASE_MAPPING_ID in mapping_ids, f"Base mapping (id={BASE_MAPPING_ID}) should be in list"
print(f"CRUD 2.2 PASSED: Base mapping in list (IDs: {mapping_ids})")

In [ ]:
# Test: Get mapping by ID
mapping = client.mappings.get(BASE_MAPPING_ID)

assert mapping.id == BASE_MAPPING_ID, f"Expected mapping id={BASE_MAPPING_ID}, got {mapping.id}"
assert mapping.name == BASE_MAPPING_NAME, f"Expected '{BASE_MAPPING_NAME}', got '{mapping.name}'"
print(f"CRUD 2.3 PASSED: Mapping retrieved: {mapping.name}")

In [ ]:
# Test: Mapping has required fields
assert mapping.id is not None, "Mapping should have id"
assert mapping.name is not None, "Mapping should have name"
assert mapping.owner_username is not None, "Mapping should have owner_username"
assert mapping.current_version is not None, "Mapping should have current_version"
assert mapping.created_at is not None, "Mapping should have created_at"

print(f"CRUD 2.4 PASSED: Mapping fields: id={mapping.id}, name={mapping.name}, version={mapping.current_version}")

In [ ]:
# Test: Get mapping version
version = client.mappings.get_version(BASE_MAPPING_ID, version=1)

assert version is not None, "Version should not be None"
assert version.version == 1, f"Expected version 1, got {version.version}"

print(f"CRUD 2.5 PASSED: Version {version.version} retrieved")

In [ ]:
# Test: Mapping version has node and edge definitions
assert version.node_definitions is not None, "Should have node_definitions"
assert len(version.node_definitions) >= 1, "Should have at least 1 node definition"

assert version.edge_definitions is not None, "Should have edge_definitions"
assert len(version.edge_definitions) >= 1, "Should have at least 1 edge definition"

print(f"CRUD 2.6 PASSED: Node definitions: {[n.label for n in version.node_definitions]}")
print(f"  Edge definitions: {[e.type for e in version.edge_definitions]}")

In [ ]:
# Test: Person node definition exists with primary_key
person_def = next((d for d in version.node_definitions if d.label == "Person"), None)
assert person_def is not None, "Should have Person node definition"
assert person_def.primary_key is not None, "Person should have primary_key"

# Test: KNOWS edge definition exists
knows_def = next((d for d in version.edge_definitions if d.type == "KNOWS"), None)
assert knows_def is not None, "Should have KNOWS edge definition"
assert knows_def.from_node == "Person", f"Expected from_node='Person', got '{knows_def.from_node}'"
assert knows_def.to_node == "Person", f"Expected to_node='Person', got '{knows_def.to_node}'"

print(f"CRUD 2.7 PASSED: Person primary_key: {person_def.primary_key}")
print(f"  KNOWS edge: {knows_def.from_node} -> {knows_def.to_node}")

In [ ]:
# Test: List mapping versions
versions = client.mappings.list_versions(BASE_MAPPING_ID)

assert versions is not None, "Versions should not be None"
assert len(versions) >= 1, "Should have at least 1 version"

version_numbers = [v.version for v in versions]
assert 1 in version_numbers, "Version 1 should exist"

print(f"CRUD 2.8 PASSED: Mapping versions: {version_numbers}")

In [ ]:
# Test: List snapshots for a specific mapping
mapping_snapshots = client.mappings.list_snapshots(BASE_MAPPING_ID)

assert mapping_snapshots is not None, "list_snapshots should not return None"
assert len(mapping_snapshots) >= 1, f"Base mapping should have at least 1 snapshot, got {len(mapping_snapshots)}"

# Verify our base snapshot is in the list
snapshot_ids = [s.id for s in mapping_snapshots]
assert BASE_SNAPSHOT_ID in snapshot_ids, f"Base snapshot (id={BASE_SNAPSHOT_ID}) should be in mapping's snapshot list"

# Verify all returned snapshots belong to this mapping
for s in mapping_snapshots:
    assert s.mapping_id == BASE_MAPPING_ID, f"Snapshot {s.id} should have mapping_id={BASE_MAPPING_ID}, got {s.mapping_id}"

print(f"CRUD 2.8b PASSED: Mapping has {len(mapping_snapshots)} snapshot(s): {snapshot_ids}")

In [ ]:
# Test: Set lifecycle config for mapping
lifecycle_mapping = client.mappings.set_lifecycle(
    BASE_MAPPING_ID,
    ttl="PT720H",  # 30 days
    inactivity_timeout="PT168H"  # 7 days
)

assert lifecycle_mapping is not None, "set_lifecycle should return mapping"
assert lifecycle_mapping.id == BASE_MAPPING_ID, "Should be same mapping"

print(f"CRUD 2.8c PASSED: Mapping lifecycle configured")
print(f"  TTL: {lifecycle_mapping.ttl}")
print(f"  Inactivity timeout: {lifecycle_mapping.inactivity_timeout}")

### Mapping CRUD Operations

In [ ]:
# Define test node and edge for CRUD tests
test_node = NodeDefinition(
    label="TestNode",
    sql="SELECT id, name FROM test_table",
    primary_key={"name": "id", "type": "INT64"},
    properties=[
        PropertyDefinition(name="name", type="STRING"),
    ]
)

test_edge = EdgeDefinition(
    type="TEST_EDGE",
    from_node="TestNode",
    to_node="TestNode",
    sql="SELECT from_id, to_id FROM test_edges",
    from_key="from_id",
    to_key="to_id",
    properties=[]
)

print(f"Test node: {test_node.label}")
print(f"Test edge: {test_edge.type}")

In [ ]:
# Test: Create a new mapping via SDK (directly, not tracked - we'll delete it ourselves)
import uuid
test_mapping_name = f"CrudTest-Mapping-{uuid.uuid4().hex[:8]}"

created_mapping = client.mappings.create(
    name=test_mapping_name,
    description="Created via SDK for E2E testing",
    node_definitions=[test_node],
    edge_definitions=[test_edge],
)

assert created_mapping is not None, "Created mapping should not be None"
assert created_mapping.id is not None, "Created mapping should have ID"
assert created_mapping.name == test_mapping_name, f"Expected '{test_mapping_name}', got '{created_mapping.name}'"
assert created_mapping.current_version == 1, f"Expected version 1, got {created_mapping.current_version}"

test_mapping_id = created_mapping.id

print(f"CRUD 2.9 PASSED: Created mapping: {created_mapping.name} (id={test_mapping_id})")

In [ ]:
# Test: Verify created mapping can be retrieved
fetched = client.mappings.get(test_mapping_id)

assert fetched.id == test_mapping_id
assert fetched.name == test_mapping_name
assert fetched.description == "Created via SDK for E2E testing"

# Verify node and edge definitions are embedded
assert len(fetched.node_definitions) == 1, f"Expected 1 node def, got {len(fetched.node_definitions)}"
assert len(fetched.edge_definitions) == 1, f"Expected 1 edge def, got {len(fetched.edge_definitions)}"

assert fetched.node_definitions[0].label == "TestNode"
assert fetched.edge_definitions[0].type == "TEST_EDGE"

print(f"CRUD 2.10 PASSED: Retrieved mapping verified: {fetched.name}")

In [ ]:
# Test: Update mapping creates new version
updated = client.mappings.update(
    test_mapping_id,
    change_description="Added second node for testing",
    node_definitions=[
        test_node,
        NodeDefinition(
            label="SecondTestNode",
            sql="SELECT id, value FROM second_table",
            primary_key={"name": "id", "type": "INT64"},
            properties=[]
        )
    ],
    edge_definitions=[test_edge],
)

assert updated.current_version == 2, f"Expected version 2, got {updated.current_version}"

print(f"CRUD 2.11 PASSED: Updated mapping to version {updated.current_version}")

In [ ]:
# Test: Verify version history
versions = client.mappings.list_versions(test_mapping_id)

assert len(versions) == 2, f"Expected 2 versions, got {len(versions)}"
version_numbers = [v.version for v in versions]
assert 1 in version_numbers and 2 in version_numbers

print(f"CRUD 2.12 PASSED: Version history: {version_numbers}")

In [ ]:
# Test: Delete mapping (cleanup)
client.mappings.delete(test_mapping_id)

# Verify deletion
try:
    client.mappings.get(test_mapping_id)
    raise AssertionError("Should have raised NotFoundError")
except NotFoundError:
    print(f"CRUD 2.13 PASSED: Mapping {test_mapping_id} deleted successfully")

## 3. Snapshots API Tests

> **NOTE: SNAPSHOT FUNCTIONALITY DISABLED**
> 
> Explicit snapshot creation (`client.snapshots.create()`) has been disabled.
> The tests below operate on snapshots that were auto-created by `create_from_mapping()`.
> Snapshot listing, getting, updating metadata, and lifecycle management still work.
> Only explicit snapshot creation is disabled.

In [ ]:
# Test: List snapshots
snapshots = client.snapshots.list()

assert len(snapshots) >= 1, f"Expected at least 1 snapshot, got {len(snapshots)}"
print(f"CRUD 3.1 PASSED: Found {len(snapshots)} snapshot(s)")

# Test: Our base snapshot is in list
snapshot_ids = [s.id for s in snapshots]
assert BASE_SNAPSHOT_ID in snapshot_ids, f"Base snapshot (id={BASE_SNAPSHOT_ID}) should be in list"

In [ ]:
# Test: Get snapshot
snapshot = client.snapshots.get(BASE_SNAPSHOT_ID)

assert snapshot.status == "ready", f"Expected status 'ready', got '{snapshot.status}'"
print(f"CRUD 3.2 PASSED: Snapshot: {snapshot.name} (status={snapshot.status})")

In [ ]:
# Test: Update snapshot metadata (name and description)
updated_snapshot = client.snapshots.update(
    BASE_SNAPSHOT_ID,
    name=f"{BASE_SNAPSHOT_NAME}-Updated",
    description="Updated description for E2E testing"
)

assert updated_snapshot is not None, "Updated snapshot should not be None"
assert updated_snapshot.id == BASE_SNAPSHOT_ID, "Should be same snapshot"
assert "Updated" in updated_snapshot.name, f"Name should be updated, got '{updated_snapshot.name}'"

print(f"CRUD 3.3 PASSED: Snapshot updated")
print(f"  Name: {updated_snapshot.name}")

# Restore original name
client.snapshots.update(BASE_SNAPSHOT_ID, name=BASE_SNAPSHOT_NAME)

In [ ]:
# Test: Set lifecycle config for snapshot
lifecycle_snapshot = client.snapshots.set_lifecycle(
    BASE_SNAPSHOT_ID,
    ttl="PT168H",  # 7 days
    inactivity_timeout="PT24H"  # 24 hours
)

assert lifecycle_snapshot is not None, "set_lifecycle should return snapshot"
assert lifecycle_snapshot.id == BASE_SNAPSHOT_ID, "Should be same snapshot"

print(f"CRUD 3.4 PASSED: Snapshot lifecycle configured")
print(f"  TTL: {lifecycle_snapshot.ttl}")
print(f"  Inactivity timeout: {lifecycle_snapshot.inactivity_timeout}")

## 4. Instances API Tests

In [ ]:
# Test: List instances
instances = client.instances.list()

assert instances is not None, "Instances should not be None"
assert len(instances) >= 1, f"Expected at least 1 instance, got {len(instances)}"
print(f"CRUD 4.1 PASSED: Found {len(instances)} instance(s)")

# Test: Our base instance is in list
instance_ids = [i.id for i in instances]
assert BASE_INSTANCE_ID in instance_ids, f"Base instance (id={BASE_INSTANCE_ID}) should be in list"

In [ ]:
# Test: Get instance by ID
instance = client.instances.get(BASE_INSTANCE_ID)

assert instance is not None, "Instance should not be None"
assert instance.id == BASE_INSTANCE_ID, f"Expected instance id={BASE_INSTANCE_ID}, got {instance.id}"
print(f"CRUD 4.2 PASSED: Instance: {instance.name} (id={instance.id})")

In [ ]:
# Test: Instance has required fields
assert instance.id is not None, "Instance should have id"
assert instance.name is not None, "Instance should have name"
assert instance.status is not None, "Instance should have status"
assert instance.snapshot_id is not None, "Instance should have snapshot_id"
assert instance.created_at is not None, "Instance should have created_at"

print(f"CRUD 4.3 PASSED: Instance fields: id={instance.id}, status={instance.status}, snapshot_id={instance.snapshot_id}")

In [ ]:
# Test: Running instance status
assert instance.status == "running", f"Expected status 'running', got '{instance.status}'"

# Test: Running instance has URL (or is running)
assert instance.instance_url is not None or instance.status == "running", \
    "Running instance should have URL or running status"

print(f"CRUD 4.4 PASSED: Instance status: {instance.status}")
print(f"  Instance URL: {instance.instance_url}")

In [ ]:
# Test: Update instance metadata (name and description)
updated_instance = client.instances.update(
    BASE_INSTANCE_ID,
    name=f"{BASE_INSTANCE_NAME}-Updated",
    description="Updated description for E2E testing"
)

assert updated_instance is not None, "Updated instance should not be None"
assert updated_instance.id == BASE_INSTANCE_ID, "Should be same instance"
assert "Updated" in updated_instance.name, f"Name should be updated, got '{updated_instance.name}'"

print(f"CRUD 4.5 PASSED: Instance updated")
print(f"  Name: {updated_instance.name}")

# Restore original name
client.instances.update(BASE_INSTANCE_ID, name=BASE_INSTANCE_NAME)

In [ ]:
# Test: Set lifecycle config for instance
# Configure custom TTL and inactivity timeout
lifecycle_instance = client.instances.set_lifecycle(
    BASE_INSTANCE_ID,
    ttl="PT48H",  # 48 hours
    inactivity_timeout="PT2H"  # 2 hours
)

assert lifecycle_instance is not None, "set_lifecycle should return instance"
assert lifecycle_instance.id == BASE_INSTANCE_ID, "Should be same instance"

print(f"CRUD 4.6 PASSED: Instance lifecycle configured")
print(f"  TTL: {lifecycle_instance.ttl}")
print(f"  Inactivity timeout: {lifecycle_instance.inactivity_timeout}")

### 4.7 Instance Termination Test

Test that an instance can be terminated and status changes appropriately.

In [ ]:
# Test 4.7: Terminate instance
# Create a separate instance for termination testing (not tracked - we'll terminate it ourselves)
import uuid
TERM_INSTANCE_NAME = f"CrudTest-TermInstance-{uuid.uuid4().hex[:8]}"

print(f"Creating instance '{TERM_INSTANCE_NAME}' for termination test...")
term_instance = client.instances.create_and_wait(
    snapshot_id=BASE_SNAPSHOT_ID,
    name=TERM_INSTANCE_NAME,
    wrapper_type=WrapperType.RYUGRAPH,
    timeout=180,
    poll_interval=5,
)
TERM_INSTANCE_ID = term_instance.id

assert term_instance.status == "running", f"Expected 'running', got '{term_instance.status}'"
print(f"Created instance {TERM_INSTANCE_ID} (status={term_instance.status})")

# Terminate the instance (immediately deletes K8s pod + DB record)
client.instances.terminate(TERM_INSTANCE_ID)
print(f"Terminated instance {TERM_INSTANCE_ID}")

# Verify instance is GONE (terminate now immediately deletes)
try:
    client.instances.get(TERM_INSTANCE_ID)
    raise AssertionError("Instance should have been deleted after termination")
except NotFoundError:
    print(f"CRUD 4.7 PASSED: Instance {TERM_INSTANCE_ID} deleted immediately after termination")

## 5. Instance Connection Tests

In [ ]:
# Test: Connect to instance via SDK
conn = client.instances.connect(instance.id)

assert conn is not None, "Connection should not be None"
print(f"CRUD 5.1 PASSED: Connected to instance {instance.id}")
print(f"  Instance URL (from SDK): {instance.instance_url}")

In [ ]:
# Test: Connection has required managers
assert hasattr(conn, 'query'), "Connection should have query method"
assert hasattr(conn, 'algo'), "Connection should have algo manager"
assert conn.algo is not None, "algo manager should not be None"
assert hasattr(conn, 'networkx'), "Connection should have networkx manager"
assert conn.networkx is not None, "networkx manager should not be None"

print("CRUD 5.2 PASSED: Connection managers verified")

In [ ]:
# Test: Connection can get status
status = conn.status()

assert status is not None, "Status should not be None"

print(f"CRUD 5.3 PASSED: Connection status: {status}")

In [ ]:
# Test: Connection context manager works
with client.instances.connect(instance.id) as ctx_conn:
    result = ctx_conn.query("MATCH (n) RETURN count(n) AS count")
    assert result is not None

print("CRUD 5.4 PASSED: Connection context manager verified")

In [ ]:
# Test: InstanceConnection.close() explicitly closes connection
test_conn = client.instances.connect(instance.id)

# Verify connection works before close
result = test_conn.query("MATCH (n) RETURN count(n) AS cnt")
assert result is not None

# Close the connection
test_conn.close()

# Verify close() doesn't crash and connection is closed
# Note: After close, subsequent queries may fail or the connection may be in a closed state
print("CRUD 5.5 PASSED: InstanceConnection.close() executed successfully")

## Summary

In [ ]:
# Cleanup is automatic via atexit!
# Resources created via ctx.mapping(), ctx.snapshot(), ctx.instance() will be cleaned up
# when the notebook exits (or when ctx.cleanup() is called explicitly)
print("Test complete - cleanup will happen automatically on exit")

In [ ]:
# Explicitly cleanup (optional - also happens on exit)
results = ctx.cleanup()
print(f"Cleaned up: {results}")

print("\n" + "="*60)
print("CRUD E2E TESTS COMPLETED!")
print("="*60)
print("\nValidated:")
print("  1. Client Initialization:")
print("    - notebook.test() with TestPersona")
print("    - Resource managers available")
print("    - Multi-persona access via ctx.with_persona()")
print("    - Resource listing")
print("  2. Mappings API:")
print("    - list, get, get_version, list_versions")
print("    - create, update, delete")
print("    - Node and edge definitions")
print("  3. Snapshots API:")
print("    - list, get, update, set_lifecycle")
print("  4. Instances API:")
print("    - list, get, update, set_lifecycle")
print("    - 4.7: Terminate instance")
print("  5. Instance Connection:")
print("    - Connection setup")
print("    - Manager availability")
print("    - Context manager")
print("\nAll test resources cleaned up via NotebookTest atexit handler")